# 평가 노트북

Drive에 저장된 **체크포인트 + 데이터**를 불러와서 정량/체감 평가를 수행합니다.
GPU 없이 CPU만으로도 실행 가능합니다.

```
Step 0: 환경 설정 (clone + pip)
Step 1: Drive에서 데이터 + 체크포인트 복원
Step 2: 정량 평가 (PESQ / STOI / SI-SNR)
Step 3: 체감 평가 (오디오 재생 + 파형/스펙트로그램)
```

> ⚠️ 커널 재시작 시 Step 0부터 다시 실행하세요.

---
## Step 0. 환경 설정

In [ ]:
import os, sys, subprocess

GITHUB_USER = 'sungmin-park-dev'
REPO_NAME   = 'tactical-speech-enhancement'
REPO_DIR    = f'/content/{REPO_NAME}'
DRIVE_ROOT  = '/content/drive/MyDrive/Colab Notebooks/ARMY Projects/2026-tactical-speech-enhancement'

# 레포 클론
if not os.path.isdir(REPO_DIR):
    token = os.getenv('GITHUB_TOKEN', '')
    if token:
        clone_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    result = subprocess.run(['git', 'clone', clone_url, REPO_DIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ clone 실패: {result.stderr.strip()}')
        raise RuntimeError('git clone 실패')
    print('✅ 레포 클론 완료')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    print('✅ 레포 최신화 완료')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'soundfile', 'librosa', 'pesq', 'pystoi', 'pyyaml', 'tqdm', 'seaborn',
     'matplotlib'],
    check=True
)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('✅ Step 0 완료')

---
## Step 1. Drive에서 데이터 + 체크포인트 복원

In [ ]:
import glob
from google.colab import drive
drive.mount('/content/drive')

# ── 데이터 복원 ──
DATA_DIR = '/content/data/processed'
DRIVE_DATA = f'{DRIVE_ROOT}/data/processed_data.tar.gz'

if not os.path.exists(DATA_DIR):
    if os.path.exists(DRIVE_DATA):
        print('데이터 복원 중...')
        !cp "{DRIVE_DATA}" /tmp/data.tar.gz
        !mkdir -p /content/data && tar -xzf /tmp/data.tar.gz -C /content/data/
        !rm /tmp/data.tar.gz
        print('✅ 데이터 복원 완료')
    else:
        print('⚠ 데이터 없음 — download_to_drive.ipynb 먼저 실행하세요')
else:
    print('데이터 이미 존재')

# ── 체크포인트 복원 ──
DRIVE_RESULTS = f'{DRIVE_ROOT}/results'
if not os.path.exists('/content/results'):
    archives = sorted(glob.glob(f'{DRIVE_RESULTS}/results_*.tar.gz'))
    if archives:
        latest = archives[-1]
        print(f'체크포인트 복원 중: {os.path.basename(latest)}')
        !cp "{latest}" /tmp/results.tar.gz
        !tar -xzf /tmp/results.tar.gz -C /content/
        !rm /tmp/results.tar.gz
        print('✅ 체크포인트 복원 완료')
    else:
        print('⚠ 체크포인트 없음 — train_colab.ipynb 먼저 실행하세요')
else:
    print('체크포인트 이미 존재')

# ── 사용 가능한 체크포인트 ──
print('\n=== 사용 가능한 체크포인트 ===')
for ckpt in sorted(glob.glob('/content/results/**/best.pt', recursive=True)):
    print(f'  {ckpt}')

---
## Step 2. 정량 평가 (PESQ / STOI / SI-SNR)

Noisy(향상 전) vs Enhanced(향상 후) 비교

In [ ]:
# ════════════════════════════════════════
# 설정 — 평가할 체크포인트 경로를 지정하세요
# ════════════════════════════════════════
CKPT_PATH = '/content/results/dpcrn/alpha_search/alpha0.3/best.pt'
ENV       = 'military'
SPLIT     = 'val'      # 'val' 또는 'test'
N_SAMPLES = 50         # 평가할 샘플 수
SR        = 16000

In [ ]:
import torch
import numpy as np
from models.dpcrn_dual import DPCRNDual
from data.dataset import SpeechEnhancementDataset
from evaluate import compute_metrics

# ── 모델 로드 ──
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
cfg = ckpt['config']
ablation_id = ckpt.get('ablation_id', 'A-soft')

model = DPCRNDual(
    n_fft=cfg['audio']['n_fft'],
    hop_length=cfg['audio']['hop_length'],
    win_length=cfg['audio']['win_length'],
    ablation_id=ablation_id,
    n_dual_path_blocks=cfg['model']['n_dual_path_blocks'],
    lstm_hidden=cfg['model']['lstm_hidden'],
    use_bwe=cfg['model'].get('use_bwe', False),
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'모델 로드: {ablation_id} | epoch {ckpt["epoch"]+1} | val_loss={ckpt["val_loss"]:.4f}')

# ── 데이터 로드 ──
mask_map = {'A-soft': 'soft', 'A-hard': 'hard', 'A-param': 'parametric'}
mask_type = mask_map.get(ablation_id, 'soft')

dataset = SpeechEnhancementDataset(
    data_dir=DATA_DIR, env=ENV, split=SPLIT, mask_type=mask_type,
)
N_SAMPLES = min(N_SAMPLES, len(dataset))
print(f'데이터: {len(dataset)}개 중 {N_SAMPLES}개 평가')

In [ ]:
# ── 추론 + 지표 계산 ──
results = {'si_snr': [], 'pesq': [], 'stoi': []}
results_noisy = {'si_snr': [], 'pesq': [], 'stoi': []}

print(f'{N_SAMPLES}개 샘플 평가 중...')
with torch.no_grad():
    for i in range(N_SAMPLES):
        sample = dataset[i]
        bc    = sample['bc'].unsqueeze(0).to(device)
        ac    = sample['ac'].unsqueeze(0).to(device)
        clean = sample['clean'].numpy()
        mask  = sample['mask'].unsqueeze(0).to(device) if ablation_id in ('A-soft','A-hard','A-param') else None
        ac_in = ac if ablation_id != 'C' else None

        enhanced = model(bc, ac_in, mask).squeeze(0).cpu().numpy()
        noisy_np = sample['ac'].numpy()

        min_len = min(len(enhanced), len(clean), len(noisy_np))
        enhanced, clean, noisy_np = enhanced[:min_len], clean[:min_len], noisy_np[:min_len]

        m = compute_metrics(enhanced, clean, sample_rate=SR)
        for k in results: results[k].append(m[k])

        m_n = compute_metrics(noisy_np, clean, sample_rate=SR)
        for k in results_noisy: results_noisy[k].append(m_n[k])

        if (i+1) % 10 == 0:
            print(f'  [{i+1}/{N_SAMPLES}]')

# ── 결과 테이블 ──
print(f'\n{"="*60}')
print(f'정량 평가 결과 ({ablation_id}, {SPLIT}, N={N_SAMPLES})')
print(f'{"="*60}')
print(f'{"":>14} | {"SI-SNR (dB)":>14} | {"PESQ":>10} | {"STOI":>10}')
print(f'{"-"*60}')

for label, res in [('Noisy (전)', results_noisy), ('Enhanced (후)', results)]:
    print(f'{label:>14} | '
          f'{np.mean(res["si_snr"]):>8.2f} ± {np.std(res["si_snr"]):4.2f} | '
          f'{np.mean(res["pesq"]):>6.3f} ± {np.std(res["pesq"]):4.2f} | '
          f'{np.mean(res["stoi"]):>6.4f} ± {np.std(res["stoi"]):5.3f}')

print(f'{"-"*60}')
d_s = np.mean(results['si_snr']) - np.mean(results_noisy['si_snr'])
d_p = np.mean(results['pesq'])   - np.mean(results_noisy['pesq'])
d_t = np.mean(results['stoi'])   - np.mean(results_noisy['stoi'])
print(f'{"개선량 (Δ)":>14} | {d_s:>+8.2f}          | {d_p:>+6.3f}      | {d_t:>+6.4f}')
print(f'\n✅ 정량 평가 완료')

---
## Step 3. 체감 평가 (오디오 재생 + 시각화)

Clean / BC / Noisy AC / Enhanced 4종을 직접 듣고, 파형 + 스펙트로그램으로 비교합니다.

In [ ]:
import IPython.display as ipd
import matplotlib.pyplot as plt
import librosa
import librosa.display

# ════════════════════════════════════════
# 설정 — 비교할 샘플 인덱스
# ════════════════════════════════════════
SAMPLE_INDICES = [0, 5, 10]

for idx in SAMPLE_INDICES:
    sample = dataset[idx]
    bc    = sample['bc'].unsqueeze(0).to(device)
    ac    = sample['ac'].unsqueeze(0).to(device)
    clean = sample['clean'].numpy()
    mask  = sample['mask'].unsqueeze(0).to(device) if ablation_id in ('A-soft','A-hard','A-param') else None
    ac_in = ac if ablation_id != 'C' else None

    with torch.no_grad():
        enhanced = model(bc, ac_in, mask).squeeze(0).cpu().numpy()

    noisy = sample['ac'].numpy()
    bc_np = sample['bc'].numpy()

    min_len = min(len(enhanced), len(clean), len(noisy))
    enhanced, clean, noisy, bc_np = enhanced[:min_len], clean[:min_len], noisy[:min_len], bc_np[:min_len]

    m_enh   = compute_metrics(enhanced, clean, sample_rate=SR)
    m_noisy = compute_metrics(noisy, clean, sample_rate=SR)

    # ── 오디오 재생 ──
    print(f'\n{"="*60}')
    print(f'샘플 #{idx}')
    print(f'{"="*60}')

    print(f'\n▶ Clean (정답)')
    ipd.display(ipd.Audio(clean, rate=SR))

    print(f'\n▶ BC (골전도 입력)')
    ipd.display(ipd.Audio(bc_np, rate=SR))

    print(f'\n▶ Noisy AC — SI-SNR={m_noisy["si_snr"]:.2f}dB, PESQ={m_noisy["pesq"]:.3f}, STOI={m_noisy["stoi"]:.4f}')
    ipd.display(ipd.Audio(noisy, rate=SR))

    print(f'\n▶ Enhanced — SI-SNR={m_enh["si_snr"]:.2f}dB, PESQ={m_enh["pesq"]:.3f}, STOI={m_enh["stoi"]:.4f}')
    ipd.display(ipd.Audio(enhanced, rate=SR))

    # ── 파형 비교 ──
    fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
    t = np.arange(min_len) / SR

    for ax, sig, label, color in zip(
        axes,
        [clean, bc_np, noisy, enhanced],
        ['Clean', 'BC (골전도)', 'Noisy AC', 'Enhanced'],
        ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
    ):
        ax.plot(t, sig, color=color, linewidth=0.3, alpha=0.8)
        ax.set_ylabel(label, fontsize=10)
        ax.set_ylim(-1, 1)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'샘플 #{idx} — 파형 비교', fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── 스펙트로그램 비교 ──
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))

    for ax, sig, label in zip(
        axes,
        [clean, bc_np, noisy, enhanced],
        ['Clean', 'BC', 'Noisy AC', 'Enhanced']
    ):
        S = librosa.stft(sig, n_fft=512, hop_length=160)
        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
        librosa.display.specshow(S_db, sr=SR, hop_length=160,
                                 x_axis='time', y_axis='hz', ax=ax,
                                 vmin=-60, vmax=0, cmap='magma')
        ax.set_title(label, fontsize=11)
        ax.set_ylim(0, 8000)

    fig.suptitle(f'샘플 #{idx} — 스펙트로그램 비교', fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── 개선량 ──
    print(f'\n  Δ SI-SNR = {m_enh["si_snr"] - m_noisy["si_snr"]:+.2f} dB')
    print(f'  Δ PESQ   = {m_enh["pesq"] - m_noisy["pesq"]:+.3f}')
    print(f'  Δ STOI   = {m_enh["stoi"] - m_noisy["stoi"]:+.4f}')

print(f'\n✅ 체감 평가 완료')